In [24]:
# Эхний cell
%pip install pyDatalog

Note: you may need to restart the kernel to use updated packages.


In [25]:
from pyDatalog import pyDatalog
pyDatalog.clear() 

# ЭНД great_grandparent гэдгийг нэмсэн:
pyDatalog.create_terms('''
    male, female, parent, person, has_child, childless,
    father, mother, son, daughter, grandparent, sibling, brother,
    sister, uncle, aunt, cousin, ancestor, descendant, great_grandparent,
    X, Y, Z, W, P, C,
    spouse, uncle_in_law, aunt_in_law
''')

def count(query_str):
    result = pyDatalog.ask(query_str)
    return len(result.answers) if result else 0

In [26]:
# Хоёр дахь cell
# Эрэгтэйчүүд
for p in ['bataa', 'bold', 'tomor', 'dorj', 'zolboo', 'munkh', 'bat', 'temuulen']:
    +male(p)

# Эмэгтэйчүүд
for p in ['saraa', 'oyuna', 'tsetseg', 'ganaa', 'anar', 'naraa', 'enkhee']:
    +female(p)

# Эцэг эхийн холбоо (Эцэг, Эх хоёроор тусад нь давталт ашиглан оруулах)
couples_children = [
    ('bataa', 'saraa', ['bold', 'oyuna', 'tomor']),
    ('bold', 'tsetseg', ['anar', 'zolboo']),
    ('dorj', 'oyuna', ['munkh', 'naraa']),
    ('tomor', 'ganaa', ['enkhee']),
    ('bat', 'anar', ['temuulen'])
]

for father_name, mother_name, children in couples_children:
    for child in children:
        +parent(father_name, child)
        +parent(mother_name, child)

# Шалгах
print(f"parent: {count('parent(X, Y)')}") # 18 гарах ёстой
print(f"male: {count('male(X)')}") # 8 гарах ёстой
print(f"female: {count('female(X)')}") # 7 гарах ёстой
print(parent('bataa', X)) # bold, oyuna, tomor гарна

parent: 18
male: 8
female: 7
X    
-----
oyuna
tomor
bold 


In [27]:
# Гурав дахь cell
# ∀x ∀y (parent(x, y) ∧ male(x) → father(x, y))
father(X, Y) <= parent(X, Y) & male(X)

# ∀x ∀y (parent(x, y) ∧ female(x) → mother(x, y))
mother(X, Y) <= parent(X, Y) & female(X)

# ∀x ∀y (parent(y, x) ∧ male(x) → son(x, y))
son(X, Y) <= parent(Y, X) & male(X)

# ∀x ∀y (parent(y, x) ∧ female(x) → daughter(x, y))
daughter(X, Y) <= parent(Y, X) & female(X)

# ∀x (male(x) ∨ female(x) → person(x))
person(X) <= male(X)
person(X) <= female(X)

# Шалгах
print(father(X, 'anar')) # bold
print(mother(X, 'bold')) # saraa
print(f"person: {count('person(X)')}") # 15

X   
----
bold
X    
-----
saraa
person: 15


In [28]:
# Дөрөв дэх cell
# ∀x ∀y (∃z (parent(x, z) ∧ parent(z, y)) → grandparent(x, y))
# Энд Z нь бие дэх оршихуйн (∃ - existential) хувьсагч юм.
grandparent(X, Y) <= parent(X, Z) & parent(Z, Y)

# ∀x ∀y (∃z (parent(z, x) ∧ parent(z, y) ∧ x ≠ y) → sibling(x, y))
sibling(X, Y) <= parent(Z, X) & parent(Z, Y) & (X != Y)

# Шалгах
print(grandparent(X, 'temuulen')) # bold, tsetseg
print(f"grandparent count: {count('grandparent(X,Y)')}") # 12
print(f"sibling count: {count('sibling(X,Y)')}") # 10

X      
-------
tsetseg
bold   
grandparent count: 12
sibling count: 10


In [29]:
# Тав дахь cell
brother(X, Y) <= sibling(X, Y) & male(X)
sister(X, Y) <= sibling(X, Y) & female(X)
uncle(X, Y) <= brother(X, Z) & parent(Z, Y)
aunt(X, Y) <= sister(X, Z) & parent(Z, Y)
cousin(X, Y) <= parent(Z, X) & sibling(Z, W) & parent(W, Y)

# Шалгах
print(f"cousins of enkhee: {cousin(X, 'enkhee')}") # 4 хүн
print(f"cousin count: {count('cousin(X, Y)')}") # 16

cousins of enkhee: X     
------
naraa 
munkh 
zolboo
anar  
cousin count: 16


In [30]:
# Зургуудугаар cell
ancestor(X, Y) <= parent(X, Y)
ancestor(X, Y) <= parent(X, Z) & ancestor(Z, Y) # parent эхэнд байх нь найдвартай
descendant(X, Y) <= ancestor(Y, X)

# Шалгах
print(f"ancestors of temuulen: {ancestor(X, 'temuulen')}") # 6 хүн
print(f"ancestor count: {count('ancestor(X,Y)')}") # 32

ancestors of temuulen: X      
-------
bataa  
saraa  
bold   
tsetseg
anar   
bat    
ancestor count: 32


In [31]:
# Долоодугаар cell
has_child(X) <= parent(X, Y)
childless(X) <= person(X) & ~has_child(X)

great_grandparent(X, Y) <= parent(X, Z) & grandparent(Z, Y)

# Шалгах
print(f"childless: {childless(X)}") # 5 хүн

childless: X       
--------
enkhee  
naraa   
temuulen
munkh   
zolboo  


In [32]:
# X, Y хоёр дундын нэг хүүхэдтэй (Z) бөгөөд өөр хүмүүс
spouse(X, Y) <= parent(X, Z) & parent(Y, Z) & (X != Y)

uncle_in_law(X, Y) <= spouse(X, Z) & aunt(Z, Y)
aunt_in_law(X, Y) <= spouse(X, Z) & uncle(Z, Y)

print(count("spouse(X, Y)")) # 10

10


In [33]:
+male('dulguun')
+parent('bat', 'dulguun')
+parent('anar', 'dulguun')

# Автоматаар шинэчлэгдсэнийг шалгах
print(sibling(X, 'temuulen')) # dulguun гарна

X      
-------
dulguun
